# **1. DATA SOURCE**

Mengambil data weather dari []

Mengambil data berita cuaca dari []

In [ ]:
# ini buat requirement installation
 
!pip install kafka-python requests feedparser
import json
import time
import hashlib
import threading
import requests
import feedparser
from datetime import datetime
from kafka import KafkaProducer, KafkaAdminClient, KafkaConsumer
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError

In [ ]:
#buat ambil data dari datasource (berita)

BOOTSTRAP_SERVERS = ["localhost:9092"]

KOTA = [
    {"kode": "JKT", "nama": "Jakarta",  "lat": -6.21, "lon": 106.85},
    {"kode": "SBY", "nama": "Surabaya", "lat": -7.25, "lon": 112.75},
    {"kode": "SMG", "nama": "Semarang", "lat": -6.99, "lon": 110.42},
    {"kode": "MDN", "nama": "Medan",    "lat": -3.59, "lon": 98.67},
    {"kode": "MKS", "nama": "Makassar", "lat": -5.14, "lon": 119.41},
    {"kode": "DPS", "nama": "Denpasar", "lat": -8.67, "lon": 115.21},
]

RSS_URLS = [
    "https://rss.tempo.co/tag/cuaca",
    "https://rss.kompas.com/feed/kompas.com/sains/environment",
]

TOPIC_API = "weather-api"
TOPIC_RSS = "weather-rss"

INTERVAL_API = 600   # 10 menit
INTERVAL_RSS = 300   # 5 menit

# Stop flag — panggil stop_flag.set() dari sel manapun untuk hentikan semua thread
stop_flag = threading.Event()

print(" Konfigurasi selesai")
print(f"   Kota dipantau : {[k['kode'] for k in KOTA]}")
print(f"   Topic API     : {TOPIC_API}")
print(f"   Topic RSS     : {TOPIC_RSS}")

 Konfigurasi selesai
   Kota dipantau : ['JKT', 'SBY', 'SMG', 'MDN', 'MKS', 'DPS']
   Topic API     : weather-api
   Topic RSS     : weather-rss


In [ ]:
#buat ambil data dari datasource (api cuaca) + tes koneksi API

def ambil_cuaca(kota: dict) -> dict | None:
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": kota["lat"],
        "longitude": kota["lon"],
        "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code",
        "timezone": "Asia/Jakarta",
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()["current"]
        return {
            "kode_kota": kota["kode"],
            "nama_kota": kota["nama"],
            "temperature": data["temperature_2m"],
            "humidity":    data["relative_humidity_2m"],
            "wind_speed":  data["wind_speed_10m"],
            "weather_code": data["weather_code"],
            "timestamp":   datetime.now().isoformat(),
        }
    except Exception as e:
        print(f"  [ERROR] {kota['nama']}: {e}")
        return None

# Test: ambil data semua kota sekali
print("Test fetch Open-Meteo:\n")
for kota in KOTA:
    event = ambil_cuaca(kota)
    if event:
        print(f"  {event['kode_kota']} | {event['nama_kota']:10s} | "
              f"{event['temperature']}°C | "
              f" {event['wind_speed']} km/h | "
              f" {event['humidity']}%")

Test fetch Open-Meteo:

  JKT | Jakarta    | 31.4°C |  12.2 km/h |  63%
  SBY | Surabaya   | 30.1°C |  6.4 km/h |  72%
  SMG | Semarang   | 32.2°C |  8.6 km/h |  57%
  MDN | Medan      | 29.0°C |  4.6 km/h |  74%
  MKS | Makassar   | 29.5°C |  7.8 km/h |  72%
  DPS | Denpasar   | 28.9°C |  15.7 km/h |  79%


In [ ]:
# buat test koneksi RSS single fetch, tidak looping

feedparser.USER_AGENT = "Mozilla/5.0 (compatible; WeatherPulse/1.0)"

def hash_url(url: str) -> str:
    return hashlib.md5(url.encode()).hexdigest()[:8]

def fetch_rss_sekali(url: str) -> list[dict]:
    hasil = []
    try:
        feed = feedparser.parse(url)
        print(f"  Status  : {feed.get('status', 'N/A')}")
        print(f"  Feed    : {feed.feed.get('title', 'tidak ada judul')}")
        print(f"  Entries : {len(feed.entries)} artikel")
        for entry in feed.entries[:3]:  # tampilkan 3 pertama
            link = entry.get("link", "")
            artikel = {
                "judul":       entry.get("title", ""),
                "link":        link,
                "ringkasan":   entry.get("summary", "")[:200],
                "waktu_terbit": entry.get("published", datetime.now().isoformat()),
                "sumber":      feed.feed.get("title", url),
                "timestamp":   datetime.now().isoformat(),
            }
            hasil.append(artikel)
    except Exception as e:
        print(f"  [ERROR] {e}")
    return hasil

# Test semua RSS URL
for url in RSS_URLS:
    print(f"\nTest RSS: {url}")
    artikel = fetch_rss_sekali(url)
    for a in artikel:
        print(f"  → {a['judul'][:70]}...")


Test RSS: https://rss.tempo.co/tag/cuaca
  Status  : 404
  Feed    : tidak ada judul
  Entries : 0 artikel

Test RSS: https://rss.kompas.com/feed/kompas.com/sains/environment
  Status  : 404
  Feed    : tidak ada judul
  Entries : 0 artikel


# **1. DATA INGEST**

In [ ]:
#buat masukin data yang diambil ke kafka dan tes koneksi Kafka

producer_api = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    key_serializer=lambda k: k.encode("utf-8"),
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    enable_idempotence=True,
    acks="all",
)

def loop_producer_api():
    print("[API Producer] Dimulai")
    while not stop_flag.is_set():
        print(f"\n[API Producer] [{datetime.now().strftime('%H:%M:%S')}] Polling...")
        for kota in KOTA:
            if stop_flag.is_set():
                break
            event = ambil_cuaca(kota)
            if event:
                producer_api.send(TOPIC_API, key=kota["kode"], value=event)
                print(f"   {event['kode_kota']} | {event['temperature']}°C | "
                      f" {event['wind_speed']} km/h |  {event['humidity']}%")
        producer_api.flush()
        # tunggu interval, tapi cek stop_flag tiap 5 detik
        for _ in range(INTERVAL_API // 5):
            if stop_flag.is_set():
                break
            time.sleep(5)
    print("[API Producer] Berhenti")

thread_api = threading.Thread(target=loop_producer_api, daemon=True, name="ProducerAPI")
thread_api.start()
print(f" Thread producer API berjalan: {thread_api.is_alive()}")

[API Producer] Dimulai

[API Producer] [15:39:51] Polling...
 Thread producer API berjalan: True


In [ ]:
#buat masukin data yang diambil ke kafka dan tes koneksi Kafka

producer_rss = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    key_serializer=lambda k: k.encode("utf-8"),
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    enable_idempotence=True,
    acks="all",
)

sudah_dikirim = set()  # deduplication

def loop_producer_rss():
    print("[RSS Producer] Dimulai")
    while not stop_flag.is_set():
        print(f"\n[RSS Producer] [{datetime.now().strftime('%H:%M:%S')}] Polling...")
        total_baru = 0
        for url in RSS_URLS:
            try:
                feed = feedparser.parse(url)
                for entry in feed.entries:
                    link = entry.get("link", "")
                    if not link or link in sudah_dikirim:
                        continue
                    artikel = {
                        "judul":        entry.get("title", ""),
                        "link":         link,
                        "ringkasan":    entry.get("summary", "")[:300],
                        "waktu_terbit": entry.get("published", datetime.now().isoformat()),
                        "sumber":       feed.feed.get("title", url),
                        "timestamp":    datetime.now().isoformat(),
                    }
                    key = hash_url(link)
                    producer_rss.send(TOPIC_RSS, key=key, value=artikel)
                    sudah_dikirim.add(link)
                    total_baru += 1
            except Exception as e:
                print(f"  [ERROR] RSS {url}: {e}")
        producer_rss.flush()
        print(f"  {total_baru} artikel baru dikirim ke {TOPIC_RSS}")
        for _ in range(INTERVAL_RSS // 5):
            if stop_flag.is_set():
                break
            time.sleep(5)
    print("[RSS Producer] Berhenti")

thread_rss = threading.Thread(target=loop_producer_rss, daemon=True, name="ProducerRSS")
thread_rss.start()
print(f" Thread producer RSS berjalan: {thread_rss.is_alive()}")

[RSS Producer] Dimulai

[RSS Producer] [15:39:51] Polling...
 Thread producer RSS berjalan: True


In [ ]:
# buat verifikasi

from kafka import TopicPartition

def peek_topic(topic: str, n: int = 5):
    consumer = KafkaConsumer(
        bootstrap_servers=BOOTSTRAP_SERVERS,
        auto_offset_reset="earliest",
        consumer_timeout_ms=5000,
        value_deserializer=lambda m: json.loads(m.decode("utf-8")),
        key_deserializer=lambda k: k.decode("utf-8") if k else None,
    )
    consumer.subscribe([topic])
    print(f"\n Isi topic '{topic}' (maks {n} event):\n")
    count = 0
    for msg in consumer:
        print(f"  offset={msg.offset} | key={msg.key} | "
              f"value={json.dumps(msg.value, ensure_ascii=False)[:120]}...")
        count += 1
        if count >= n:
            break
    if count == 0:
        print("  (kosong — tunggu producer polling pertama selesai)")
    consumer.close()

peek_topic(TOPIC_API)
peek_topic(TOPIC_RSS)


 Isi topic 'weather-api' (maks 5 event):

  offset=0 | key=MKS | value={"kode_kota": "MKS", "nama_kota": "Makassar", "temperature": 29.8, "humidity": 72, "wind_speed": 7.0, "weather_code": 3,...
  offset=1 | key=MKS | value={"kode_kota": "MKS", "nama_kota": "Makassar", "temperature": 29.7, "humidity": 72, "wind_speed": 7.4, "weather_code": 3,...
  offset=2 | key=MKS | value={"kode_kota": "MKS", "nama_kota": "Makassar", "temperature": 29.5, "humidity": 72, "wind_speed": 7.8, "weather_code": 3,...
  offset=0 | key=SBY | value={"kode_kota": "SBY", "nama_kota": "Surabaya", "temperature": 30.5, "humidity": 71, "wind_speed": 6.4, "weather_code": 3,...
  offset=1 | key=SMG | value={"kode_kota": "SMG", "nama_kota": "Semarang", "temperature": 33.3, "humidity": 51, "wind_speed": 9.6, "weather_code": 3,...

 Isi topic 'weather-rss' (maks 5 event):

  0 artikel baru dikirim ke weather-rss
   JKT | 31.4°C |  12.2 km/h |  63%
   SBY | 30.1°C |  6.4 km/h |  72%
   SMG | 32.2°C |  8.6 km/h |  57%
   

In [ ]:
# setup folder lokal untuk dashboard
import os

os.makedirs("dashboard/data", exist_ok=True)

for fname in ["live_api.json", "live_rss.json"]:
    fpath = f"dashboard/data/{fname}"
    if not os.path.exists(fpath):
        with open(fpath, "w") as f:
            json.dump([], f)
        print(f"  ✅ Dibuat: {fpath}")
    else:
        print(f"  ℹ️  Sudah ada: {fpath}")

print("
Folder dashboard/data siap.")

In [ ]:
# Consumer: baca dari weather-api + weather-rss → simpan ke HDFS & lokal
# Buffer dikosongkan setiap FLUSH_INTERVAL detik

import subprocess

FLUSH_INTERVAL = 120   # simpan ke HDFS tiap 2 menit
HDFS_API_PATH  = "/data/weather/api"
HDFS_RSS_PATH  = "/data/weather/rss"
LOCAL_API_FILE = "dashboard/data/live_api.json"
LOCAL_RSS_FILE = "dashboard/data/live_rss.json"

buffer_api = []
buffer_rss = []
lock_api   = threading.Lock()
lock_rss   = threading.Lock()

def simpan_ke_hdfs(data: list, hdfs_path: str, local_file: str, label: str):
    if not data:
        return
    ts        = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    tmp_file  = f"/tmp/weather_{label}_{ts}.json"
    hdfs_dest = f"{hdfs_path}/{ts}.json"

    # Tulis ke file lokal sementara
    with open(tmp_file, "w") as f:
        json.dump(data, f, ensure_ascii=False)

    # Copy file ke dalam container lalu hdfs put
    subprocess.run(["docker", "cp", tmp_file, f"hadoop-namenode:{tmp_file}"], capture_output=True)
    result = subprocess.run(
        ["docker", "exec", "hadoop-namenode",
         "hdfs", "dfs", "-put", "-f", tmp_file, hdfs_dest],
        capture_output=True, text=True
    )

    if result.returncode == 0:
        print(f"  [{label}] 📦 HDFS: {hdfs_dest} ({len(data)} event)")
    else:
        print(f"  [{label}] ❌ Gagal HDFS: {result.stderr.strip()}")

    # Simpan salinan lokal untuk dashboard (50 event terbaru)
    with open(local_file, "w") as f:
        json.dump(data[-50:], f, ensure_ascii=False)

    os.remove(tmp_file)

def loop_consumer_api():
    consumer = KafkaConsumer(
        TOPIC_API,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id="consumer-hdfs-api",
        auto_offset_reset="earliest",
        value_deserializer=lambda m: json.loads(m.decode("utf-8")),
        consumer_timeout_ms=2000,
    )
    print("[Consumer API] Dimulai")
    last_flush = time.time()
    while not stop_flag.is_set():
        try:
            for msg in consumer:
                with lock_api:
                    buffer_api.append(msg.value)
                if stop_flag.is_set():
                    break
            if time.time() - last_flush >= FLUSH_INTERVAL:
                with lock_api:
                    data_copy = buffer_api.copy()
                    buffer_api.clear()
                simpan_ke_hdfs(data_copy, HDFS_API_PATH, LOCAL_API_FILE, "API")
                last_flush = time.time()
        except Exception as e:
            print(f"  [Consumer API] Error: {e}")
            time.sleep(5)
    # flush sisa buffer saat stop
    with lock_api:
        data_copy = buffer_api.copy()
        buffer_api.clear()
    simpan_ke_hdfs(data_copy, HDFS_API_PATH, LOCAL_API_FILE, "API")
    consumer.close()
    print("[Consumer API] Berhenti")

def loop_consumer_rss():
    consumer = KafkaConsumer(
        TOPIC_RSS,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id="consumer-hdfs-rss",
        auto_offset_reset="earliest",
        value_deserializer=lambda m: json.loads(m.decode("utf-8")),
        consumer_timeout_ms=2000,
    )
    print("[Consumer RSS] Dimulai")
    last_flush = time.time()
    while not stop_flag.is_set():
        try:
            for msg in consumer:
                with lock_rss:
                    buffer_rss.append(msg.value)
                if stop_flag.is_set():
                    break
            if time.time() - last_flush >= FLUSH_INTERVAL:
                with lock_rss:
                    data_copy = buffer_rss.copy()
                    buffer_rss.clear()
                simpan_ke_hdfs(data_copy, HDFS_RSS_PATH, LOCAL_RSS_FILE, "RSS")
                last_flush = time.time()
        except Exception as e:
            print(f"  [Consumer RSS] Error: {e}")
            time.sleep(5)
    with lock_rss:
        data_copy = buffer_rss.copy()
        buffer_rss.clear()
    simpan_ke_hdfs(data_copy, HDFS_RSS_PATH, LOCAL_RSS_FILE, "RSS")
    consumer.close()
    print("[Consumer RSS] Berhenti")

thread_consumer_api = threading.Thread(target=loop_consumer_api, daemon=True, name="ConsumerAPI")
thread_consumer_rss = threading.Thread(target=loop_consumer_rss, daemon=True, name="ConsumerRSS")
thread_consumer_api.start()
thread_consumer_rss.start()

print(f"✅ Consumer API berjalan : {thread_consumer_api.is_alive()}")
print(f"✅ Consumer RSS berjalan : {thread_consumer_rss.is_alive()}")
print(f"
⏱️  Flush ke HDFS setiap {FLUSH_INTERVAL} detik")

In [ ]:
# Verifikasi: cek file yang sudah masuk ke HDFS
# Jalankan setelah minimal 1 flush interval (2 menit)

def cek_hdfs(path):
    result = subprocess.run(
        ["docker", "exec", "hadoop-namenode",
         "hdfs", "dfs", "-ls", path],
        capture_output=True, text=True
    )
    print(f"
📂 {path}:")
    print(result.stdout if result.stdout else "  (kosong — tunggu flush berikutnya)")

cek_hdfs("/data/weather/api")
cek_hdfs("/data/weather/rss")

print("📄 Salinan lokal dashboard:")
for fname in ["live_api.json", "live_rss.json"]:
    fpath = f"dashboard/data/{fname}"
    if os.path.exists(fpath):
        with open(fpath) as f:
            data = json.load(f)
        print(f"  {fpath}: {len(data)} event terbaru")
    else:
        print(f"  {fpath}: belum ada")

In [ ]:
# cek status semua thread

threads = {
    "ProducerAPI" : thread_api,
    "ProducerRSS" : thread_rss,
    "ConsumerAPI" : thread_consumer_api,
    "ConsumerRSS" : thread_consumer_rss,
}

print("Status thread:
")
for nama, t in threads.items():
    status = "🟢 berjalan" if t.is_alive() else "🔴 mati"
    print(f"  {nama:15s} : {status}")

In [ ]:
# stop dan tutup semua koneksi

stop_flag.set()
time.sleep(3)

producer_api.close()
producer_rss.close()

print("✅ Semua producer & consumer dihentikan")
for nama, t in threads.items():
    print(f"  {nama:15s} : {'mati' if not t.is_alive() else 'masih jalan'}")

# Reset stop_flag kalau mau jalankan ulang
# stop_flag.clear()

# **1. DATA STORE**

# **1. DATA ANALYSIS**

# **1. DATA SERVE**